# Pocket Agent — Qwen3-0.6B Fine-tune

Fine-tune `unsloth/Qwen3-0.6B` on synthetic tool-calling data, merge LoRA weights back into the base, then quantize to GGUF.

## Install

In [2]:
!pip install "unsloth>=2026.4.4" "transformers>=5.3.0,<6.0.0" \
"trl>=0.24.0" "peft>=0.13.0" "accelerate>=1.0.0" \
"bitsandbytes>=0.44.0" "datasets>=3.0.0" "torch>=2.6.0" \
"llama-cpp-python>=0.3.0" "gradio>=5.0.0" \
"numpy>=1.26.0" sentencepiece protobuf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 MB 33.3 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 14

## Config

In [22]:
CFG = {
    "base_model":       "unsloth/Qwen3-0.6B",
    "train_file":       "/kaggle/input/datasets/tayyabahmad44/cleaned-dataset/train_qwen_cleaned.jsonl",
    "output_dir":       "artifacts/lora_adapter",
    "merged_dir":       "artifacts/merged_16bit",
    "max_seq_length":   1536,
    "lora_r":           32,
    "lora_alpha":       64,
    "lora_dropout":     0.0,
    "num_epochs":       3,
    "lr":               2e-4,
    "per_device_batch": 2,
    "grad_accum":       8,
    "warmup_ratio":     0.05,
    "seed":             3407,
    "eval_ratio":       0.05,
    "load_in_4bit":     True,
}

## Imports

In [23]:
import json, os, random
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])

os.makedirs(CFG["output_dir"], exist_ok=True)
os.makedirs(CFG["merged_dir"], exist_ok=True)

## Model + LoRA

In [24]:
print(f"[model] loading {CFG['base_model']} (4bit={CFG['load_in_4bit']})")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["base_model"],
    max_seq_length=CFG["max_seq_length"],
    load_in_4bit=CFG["load_in_4bit"],
    dtype=None,  # auto — bf16 on Ampere+, fp16 on T4
)

# Qwen3 uses <|im_end|> as eos — reuse it as pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[model] attaching LoRA (r={CFG['lora_r']}, alpha={CFG['lora_alpha']})")
model = FastLanguageModel.get_peft_model(
    model,
    r=CFG["lora_r"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # ~30% less VRAM
    random_state=CFG["seed"],
    use_rslora=False,
    loftq_config=None,
)

[model] loading unsloth/Qwen3-0.6B (4bit=True)
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[model] attaching LoRA (r=32, alpha=64)


## Data

In [26]:
train_path = Path(CFG["train_file"])
assert train_path.exists(), f"Training file not found: {train_path.resolve()}"

rows = []
with train_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        assert "messages" in obj, f"Line {i}: missing 'messages'"
        rows.append({"messages": obj["messages"]})
print(f"[data] loaded {len(rows)} conversations from {train_path}")

raw_ds = Dataset.from_list(rows)

def _apply_template(ex):
    # add_generation_prompt=False coz last turn is already assistant
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False,
    )}

ds = raw_ds.map(_apply_template, remove_columns=raw_ds.column_names,
                desc="applying chat template")

if CFG["eval_ratio"] > 0 and len(ds) >= 20:
    split = ds.train_test_split(test_size=CFG["eval_ratio"], seed=CFG["seed"])
    train_ds, eval_ds = split["train"], split["test"]
else:
    train_ds, eval_ds = ds, None
print(f"[data] train={len(train_ds)} eval={len(eval_ds) if eval_ds else 0}")

[data] loaded 406 conversations from /kaggle/input/datasets/tayyabahmad44/cleaned-dataset/train_qwen_cleaned.jsonl


applying chat template:   0%|          | 0/406 [00:00<?, ? examples/s]

[data] train=385 eval=21


## Trainer

In [27]:
use_bf16 = is_bfloat16_supported()
print(f"[train] precision: {'bf16' if use_bf16 else 'fp16'}")

sft_config = SFTConfig(
    output_dir=CFG["output_dir"],
    num_train_epochs=CFG["num_epochs"],
    per_device_train_batch_size=CFG["per_device_batch"],
    gradient_accumulation_steps=CFG["grad_accum"],
    per_device_eval_batch_size=CFG["per_device_batch"],
    learning_rate=CFG["lr"],
    warmup_ratio=CFG["warmup_ratio"],
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=5,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    eval_strategy="steps" if eval_ds else "no",
    eval_steps=25,
    load_best_model_at_end=bool(eval_ds),
    metric_for_best_model="eval_loss" if eval_ds else None,
    greater_is_better=False,
    seed=CFG["seed"],
    report_to="none",
    bf16=use_bf16,
    fp16=not use_bf16,
    fp16_full_eval=not use_bf16,
    max_length=CFG["max_seq_length"],
    dataset_text_field="text",
    packing=False,               # packing breaks response-only masking
    dataset_num_proc=2,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] if eval_ds else [],
)

# only train on assistant responses, not system/user turns
# Qwen3 uses ChatML markers:
#   instruction: "<|im_start|>user\n"
#   response:    "<|im_start|>assistant\n"
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# quick check masking is working
_ex = trainer.train_dataset[0]
_labels, _ids = _ex["labels"], _ex["input_ids"]
_supervised = sum(1 for t in _labels if t != -100)
print(f"[sanity] example 0: {_supervised}/{len(_labels)} tokens supervised "
      f"({100 * _supervised / len(_labels):.1f}%)")
_supervised_ids = [tid for tid, lbl in zip(_ids, _labels) if lbl != -100]
print(f"[sanity] supervised text (example 0):\n---\n"
      f"{tokenizer.decode(_supervised_ids)}\n---")
assert _supervised > 0, (
    "train_on_responses_only masked EVERYTHING — markers don't match. "
    "Check instruction_part / response_part."
)

if torch.cuda.is_available():
    print(f"[train] GPU mem before: {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

stats = trainer.train()
print(f"[train] final loss: {stats.training_loss:.4f}")

if torch.cuda.is_available():
    print(f"[train] peak GPU mem: {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[train] precision: fp16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/385 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/21 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/385 [00:00<?, ? examples/s]

...

[sanity] example 0: 31/317 tokens supervised (9.8%)
[sanity] supervised text (example 0):
---
<think>

</think>

<tool_call>
{"tool": "weather", "args": {"location": "Cairo", "unit": "F"}}
</tool_call><|im_end|>

---
[train] GPU mem before: 1.72 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 385 | Num Epochs = 3 | Total steps = 75
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 20,185,088 of 616,235,008 (3.28% trained)


Step,Training Loss,Validation Loss
25,0.049192,0.028494
50,0.025584,0.018745
75,0.028140,0.016221


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

[train] final loss: 0.0807
[train] peak GPU mem: 2.15 GB


## Save

In [28]:
print(f"[save] adapter -> {CFG['output_dir']}")
model.save_pretrained(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])

# merged 16-bit needed as input for GGUF quantization
print(f"[save] merged 16-bit -> {CFG['merged_dir']}")
model.save_pretrained_merged(CFG["merged_dir"], tokenizer, save_method="merged_16bit")

print("[done] training complete.")

[save] adapter -> artifacts/lora_adapter
[save] merged 16-bit -> artifacts/merged_16bit
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 5184.55it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:08<00:00,  8.53s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/artifacts/merged_16bit`
[done] training complete.


## Prep Dataset

Inject `/no_think` into system prompts so Qwen3 skips the thinking chain at inference time.

In [29]:
import json
from pathlib import Path

src = Path("/kaggle/input/datasets/tayyabahmad44/synthetic/train.jsonl")
dst = Path("data/train_qwen.jsonl")
dst.parent.mkdir(exist_ok=True)

n = 0
with src.open() as f_in, dst.open("w") as f_out:
    for line in f_in:
        obj = json.loads(line)
        for msg in obj["messages"]:
            if msg["role"] == "system" and "/no_think" not in msg["content"]:
                msg["content"] = msg["content"].rstrip() + "\n/no_think"
        f_out.write(json.dumps(obj) + "\n")
        n += 1
print(f"rewrote {n} examples -> {dst}")

rewrote 406 examples -> data/train_qwen.jsonl


## Merge LoRA into Base

In [30]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model = "unsloth/Qwen3-0.6B"

# load base on CPU — bypassing unsloth coz we just need plain HF weights
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="cpu"
)
tokenizer = AutoTokenizer.from_pretrained(base_model)

# attach the trained adapter
model = PeftModel.from_pretrained(model, "artifacts/lora_adapter")

# bake adapter weights into the base model
print("Merging weights...")
model = model.merge_and_unload()

print("Saving merged model...")
model.save_pretrained("artifact/merged_16bit", safe_serialization=True)
tokenizer.save_pretrained("artifacts/merged_16bit")

print("Done — ready for quantization.")


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Merging weights...
Saving merged model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done — ready for quantization.


In [31]:
model = PeftModel.from_pretrained(
    model,
    "artifacts/lora_adapter"
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [32]:
model = model.merge_and_unload()

In [33]:
model.save_pretrained("merged_model")
tokenizer.save_pretrained("merged_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('merged_model/tokenizer_config.json',
 'merged_model/chat_template.jinja',
 'merged_model/tokenizer.json')

In [34]:
import torch

# move fully to CPU before saving
model = model.to("cpu")

# then save properly
model.save_pretrained("merged_model", safe_serialization=True)
tokenizer.save_pretrained("merged_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('merged_model/tokenizer_config.json',
 'merged_model/chat_template.jinja',
 'merged_model/tokenizer.json')

## Check Artifacts

In [35]:
import os
os.listdir("artifacts/merged_16bit")

['tokenizer.json',
 '.cache',
 'model.safetensors',
 'tokenizer_config.json',
 'config.json',
 'chat_template.jinja']

## Quantize → GGUF

In [36]:
# ============================================================================ #
# Pocket-Agent quantize                                                        #
# ============================================================================ #

CFG = {
    "merged_dir":   "artifact/merged_16bit",
    "out_path":      "artifact/pocket-agent.gguf",
    "quant":         "q4_k_m",
    "max_size_mb":  500,
    "no_fallback":  False,
}

import os, subprocess, sys, shutil, json, re
from pathlib import Path

LLAMA_CPP_DIR = Path("llama.cpp")
QUANT_FALLBACK_CHAIN = ["q5_k_m", "q4_k_m", "q4_k_s", "q3_k_m", "q3_k_s", "q2_k"]

def run(cmd):
    print(f"$ {' '.join(str(c) for c in cmd)}")
    subprocess.run(cmd, check=True)

def size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024)

# ---------- 1. Build llama.cpp (Standard) ----------
convert_script = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
quantize_bin   = LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize"

if not (convert_script.exists() and quantize_bin.exists()):
    print("[llama.cpp] cloning + building...")
    if not LLAMA_CPP_DIR.exists():
        run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp"])
    run(["apt-get", "install", "-y", "-q", "cmake", "build-essential", "libcurl4-openssl-dev"])
    run(["cmake", "-S", str(LLAMA_CPP_DIR), "-B", str(LLAMA_CPP_DIR / "build"),
         "-DBUILD_SHARED_LIBS=OFF", "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF"])
    run(["cmake", "--build", str(LLAMA_CPP_DIR / "build"), "--config", "Release", "-j", "--target", "llama-quantize"])
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(LLAMA_CPP_DIR / "requirements.txt")])
else:
    print("[llama.cpp] already built")

# ---------- 2. COMPATIBILITY PATCHES (The Fix) ----------
merged = Path(CFG["merged_dir"])
print("[patch] applying Qwen3 -> Qwen2 compatibility patches...")

# Patch 1: config.json
config_path = merged / "config.json"
if config_path.exists():
    with open(config_path, "r") as f:
        config = json.load(f)
    config["model_type"] = "qwen2" # Force architecture to qwen2
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)

# Patch 2: tokenizer_config.json
tok_config_path = merged / "tokenizer_config.json"
if tok_config_path.exists():
    with open(tok_config_path, "r") as f:
        tok_cfg = json.load(f)
    tok_cfg["model_type"] = "qwen2"
    with open(tok_config_path, "w") as f:
        json.dump(tok_cfg, f, indent=2)

# Patch 3: Force the Python script to bypass the "BPE not recognized" error
# We literally edit the script to return "qwen2" instead of crashing.
if convert_script.exists():
    with open(convert_script, "r") as f:
        script_content = f.read()
    
    old_error = 'raise NotImplementedError("BPE pre-tokenizer was not recognized - update get_vocab_base_pre()")'
    new_logic = 'return "qwen2" # Patched by Pocket-Agent'
    
    if old_error in script_content:
        script_content = script_content.replace(old_error, new_logic)
        with open(convert_script, "w") as f:
            f.write(script_content)
        print("[patch] Applied brute-force tokenizer fix to llama.cpp script")

# ---------- 3. Convert & Quantize ----------
out_path = Path(CFG["out_path"])
out_path.parent.mkdir(parents=True, exist_ok=True)
f16_path = out_path.with_name(out_path.stem + ".f16.gguf")

if not f16_path.exists():
    print(f"[quant] converting {merged} -> {f16_path}")
    run([sys.executable, str(convert_script), str(merged),
         "--outfile", str(f16_path), "--outtype", "f16"]) 
else:
    print(f"[quant] reusing existing {f16_path}")

chain = [CFG["quant"]] if CFG["no_fallback"] else (
    [CFG["quant"]] + [q for q in QUANT_FALLBACK_CHAIN if q != CFG["quant"]]
)

final_size = None
for method in chain:
    print(f"\n[quant] trying {method}")
    run([str(quantize_bin), str(f16_path), str(out_path), method])
    sz = size_mb(out_path)
    print(f"[quant] {method}: {sz:.1f} MB  (gate: {CFG['max_size_mb']} MB)")
    if sz <= CFG["max_size_mb"]:
        final_size = sz
        print(f"[quant] ✅ PASSED")
        break

if final_size:
    try: f16_path.unlink()
    except: pass
    print(f"\nFinal GGUF: {out_path} ({final_size:.1f} MB)")

[llama.cpp] already built
[patch] applying Qwen3 -> Qwen2 compatibility patches...
[quant] converting artifact/merged_16bit -> artifact/pocket-agent.f16.gguf
$ /usr/bin/python3 llama.cpp/convert_hf_to_gguf.py artifact/merged_16bit --outfile artifact/pocket-agent.f16.gguf --outtype f16


INFO:hf-to-gguf:Loading model: merged_16bit
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: Qwen3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1024, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1024, 3072}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1024, 3072}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.attn_k_norm.weight,  torch.float16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {1024, 1024}
INFO:h


[quant] trying q4_k_m
$ llama.cpp/build/bin/llama-quantize artifact/pocket-agent.f16.gguf artifact/pocket-agent.gguf q4_k_m


llama_print_build_info: build = 1 (d05fe1d)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'artifact/pocket-agent.f16.gguf' to 'artifact/pocket-agent.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 25 key-value pairs and 310 tensors from artifact/pocket-agent.f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.600000
llama_model_loader: - kv   5:                               general.na


main: quantize time = 28371.02 ms
main:    total time = 28371.02 ms
[quant] q4_k_m: 375.9 MB  (gate: 500 MB)
[quant] ✅ PASSED

Final GGUF: artifact/pocket-agent.gguf (375.9 MB)


# Inference

In [38]:
import os
from pathlib import Path
from typing import List, Dict, Optional

from llama_cpp import Llama


# --------------------------------------------------------------------------- #
# System prompt — must match what the training data used verbatim. If you   #
# change the system prompt you'll degrade tool-call accuracy on adversarial  #
# paraphrases because the model learned to condition on this exact text.    #
# --------------------------------------------------------------------------- #

SYSTEM_PROMPT = (
    "You are Pocket-Agent, a compact on-device assistant.\n\n"
    "You have access to exactly five tools. When the user's request clearly "
    "maps to one of them, respond ONLY with a tool call wrapped in "
    "<tool_call>...</tool_call> tags containing valid JSON. When no tool fits, "
    "respond with plain natural language — no tags, no JSON.\n\n"
    "Available tools:\n"
    '{"tool": "weather",  "args": {"location": "string", "unit": "C|F"}}\n'
    '{"tool": "calendar", "args": {"action": "list|create", "date": "YYYY-MM-DD", "title": "string?"}}\n'
    '{"tool": "convert",  "args": {"value": "number", "from_unit": "string", "to_unit": "string"}}\n'
    '{"tool": "currency", "args": {"amount": "number", "from": "ISO3", "to": "ISO3"}}\n'
    '{"tool": "sql",      "args": {"query": "string"}}\n\n'
    "Rules:\n"
    "- Emit EXACTLY ONE <tool_call>...</tool_call> block for tool requests, nothing else.\n"
    "- For calendar \"list\", title is optional and may be omitted.\n"
    "- For refusals, write a brief helpful plain-text reply.\n"
    "- In multi-turn conversations, resolve pronouns/references from prior turns.\n"
)


# --------------------------------------------------------------------------- #
# Model loader — cached at module level. First call pays the load cost;      #
# subsequent calls (and the 20 grading examples) reuse the same Llama.      #
# --------------------------------------------------------------------------- #

_MODEL: Optional[Llama] = None
_MODEL_PATH_ENV = "POCKET_AGENT_GGUF"
_DEFAULT_GGUF = "/kaggle/working/artifacts/pocket-agent.gguf"


def _get_model() -> Llama:
    global _MODEL
    if _MODEL is not None:
        return _MODEL

    model_path = os.environ.get(_MODEL_PATH_ENV, str(_DEFAULT_GGUF))
    if not Path(model_path).exists():
        raise FileNotFoundError(
            f"GGUF not found at {model_path}. Set {_MODEL_PATH_ENV} or place "
            f"the file at {_DEFAULT_GGUF}."
        )

    # Tuning for the Colab CPU runtime latency gate (≤200 ms/turn):
    #   - n_ctx=2048: enough for system + 3 turns, avoids wasted KV allocation
    #   - n_threads = physical cores (Colab CPU runtime is typically 2)
    #   - n_batch=512: fast prompt-prefill on short prompts
    #   - use_mmap=True: lets the OS page-cache the weights across calls
    _MODEL = Llama(
        model_path=model_path,
        n_ctx=2048,
        n_threads=max(1, (os.cpu_count() or 2)),
        n_batch=512,
        use_mmap=True,
        use_mlock=False,
        verbose=False,
        chat_format="chatml",   # Qwen3 uses ChatML; matches training
    )
    return _MODEL


# --------------------------------------------------------------------------- #
# Public API                                                                  #
# --------------------------------------------------------------------------- #

def _normalize_history(history: List[Dict]) -> List[Dict]:
    """
    Accept either OpenAI-style messages ({"role": "...", "content": "..."})
    or tuples, and strip any pre-existing system turn — we always prepend
    our own canonical SYSTEM_PROMPT so the model sees the format it trained on.
    """
    clean: List[Dict] = []
    for turn in history or []:
        if isinstance(turn, dict) and "role" in turn and "content" in turn:
            if turn["role"] == "system":
                continue  # we inject our own
            clean.append({"role": turn["role"], "content": str(turn["content"])})
    return clean


def _postprocess(text: str) -> str:
    """
    The model was trained to emit EXACTLY ONE <tool_call> block or plain text.
    In practice, small models occasionally:
      - add whitespace or newlines around the block
      - emit stray text after </tool_call>
      - emit the block at the very end of a longer reply (rare at inference)
    We strip to the first complete tool_call if present, else return trimmed text.
    """
    text = text.strip()
    start = text.find("<tool_call>")
    end = text.find("</tool_call>")
    if start != -1 and end != -1 and end > start:
        return text[start : end + len("</tool_call>")]
    return text


def run(prompt: str, history: List[Dict]) -> str:
    """
    Grader contract. `history` is prior turns (user + assistant alternating);
    `prompt` is the current user message. Returns the assistant's reply as a
    raw string.
    """
    model = _get_model()

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(_normalize_history(history))
    messages.append({"role": "user", "content": prompt})

    # Deterministic decoding for tool-call accuracy. Tool calls are a
    # structured-output problem — any sampling noise hurts arg fidelity.
    out = model.create_chat_completion(
        messages=messages,
        max_tokens=256,
        temperature=0.0,
        top_p=1.0,
        stop=["<|im_end|>"],
    )
    text = out["choices"][0]["message"]["content"] or ""
    return _postprocess(text)


# ---------------------
# Manual smoke test 
# ---------------------
tests = [
    ("What's the weather in Karachi in Celsius?", []),
    ("Convert 5 km to miles", []),
    (
        "Now convert that to meters",
        [
            {"role": "user", "content": "Convert 5 km to miles"},
            {
                "role": "assistant",
                "content": (
                    '<tool_call>{"tool":"convert","args":'
                    '{"value":5,"from_unit":"km","to_unit":"miles"}}</tool_call>'
                ),
            },
        ],
    ),
    ("What's your favorite color?", []),
]
for p, h in tests:
    print(f"\nUSER: {p}")
    print(f"BOT : {run(p, h)}")


USER: What's the weather in Karachi in Celsius?


llama_context: n_ctx_seq (2048) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


BOT : <tool_call>
{"tool": "weather", "args": {"location": "Karachi", "unit": "C"}}
</tool_call>

USER: Convert 5 km to miles
BOT : <tool_call>
{"tool": "convert", "args": {"value": 5, "from_unit": "km", "to_unit": "miles"}}
</tool_call>

USER: Now convert that to meters
BOT : <tool_call>{"tool":"convert","args":{"value":5,"from_unit":"km","to_unit":"meters"}}</tool_call>

USER: What's your favorite color?
BOT : <think>

</think>

I'm Pocket-Agent, a tool-calling assistant. I can help with weather, calendar, unit conversion, currency exchange, and SQL queries. What would you like to do?
